In [1]:
import pandas as pd


df = pd.read_excel("data/Online Retail.xlsx")
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France


In [2]:
cleaned_df = df.drop(columns=["Country", "CustomerID", "UnitPrice", "InvoiceDate", "StockCode", "Quantity"]).dropna()
cleaned_df["Description"] = cleaned_df["Description"].astype(str).apply(str.strip)
cleaned_df

,InvoiceNo,Description
0,536365,WHITE HANGING HEART T-LIGHT HOLDER
1,536365,WHITE METAL LANTERN
2,536365,CREAM CUPID HEARTS COAT HANGER
3,536365,KNITTED UNION FLAG HOT WATER BOTTLE
4,536365,RED WOOLLY HOTTIE WHITE HEART.
...,...,...
541904,581587,PACK OF 20 SPACEBOY NAPKINS
541905,581587,CHILDREN'S APRON DOLLY GIRL
541906,581587,CHILDRENS CUTLERY DOLLY GIRL
541907,581587,CHILDRENS CUTLERY CIRCUS PARADE


### Apriori rules

In [3]:
from apriori import Apriori


min_supp = 0.02
min_conf = 0.6

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", 0,
):
    rules_df = Apriori.create_rules(cleaned_df, min_supp, min_conf, "InvoiceNo", "Description")
    display(rules_df)

,A,B,support,confidence
0,"{ROSES REGENCY TEACUP AND SAUCER, PINK REGENCY TEACUP AND SAUCER}",{GREEN REGENCY TEACUP AND SAUCER},0.022458,0.894137
1,"{GREEN REGENCY TEACUP AND SAUCER, PINK REGENCY TEACUP AND SAUCER}",{ROSES REGENCY TEACUP AND SAUCER},0.022458,0.852484
2,{PINK REGENCY TEACUP AND SAUCER},{GREEN REGENCY TEACUP AND SAUCER},0.026344,0.803995
3,{PINK REGENCY TEACUP AND SAUCER},{ROSES REGENCY TEACUP AND SAUCER},0.025117,0.766542
4,{GREEN REGENCY TEACUP AND SAUCER},{ROSES REGENCY TEACUP AND SAUCER},0.032071,0.741722
5,{GARDENERS KNEELING PAD CUP OF TEA},{GARDENERS KNEELING PAD KEEP CALM},0.022458,0.717647
6,"{GREEN REGENCY TEACUP AND SAUCER, ROSES REGENCY TEACUP AND SAUCER}",{PINK REGENCY TEACUP AND SAUCER},0.022458,0.700255
7,{ROSES REGENCY TEACUP AND SAUCER},{GREEN REGENCY TEACUP AND SAUCER},0.032071,0.700000
8,{CHARLOTTE BAG PINK POLKADOT},{RED RETROSPOT CHARLOTTE BAG},0.021517,0.692105
9,{PINK REGENCY TEACUP AND SAUCER},"{GREEN REGENCY TEACUP AND SAUCER, ROSES REGENCY TEACUP AND SAUCER}",0.022458,0.685393


In [4]:
basket = set(["ROSES REGENCY TEACUP AND SAUCER", "PINK REGENCY TEACUP AND SAUCER"])

rules_df[rules_df["A"] == basket]

,A,B,support,confidence
0,"{ROSES REGENCY TEACUP AND SAUCER, PINK REGENCY...",{GREEN REGENCY TEACUP AND SAUCER},0.022458,0.894137


### FP-Growth

In [ ]:
from mlxtend.frequent_patterns import fpgrowth, association_rules as ml_association_rules


min_supp = 0.02
min_conf = 0.6

basket_matrix = (
    cleaned_df.assign(value=1)
    .pivot_table(
        index="InvoiceNo",
        columns="Description",
        values="value",
        aggfunc="max",
        fill_value=0,
    )
    .astype(bool)
)

frequent_itemsets_fp = fpgrowth(basket_matrix, min_support=min_supp, use_colnames=True)
rules_fp = ml_association_rules(frequent_itemsets_fp, metric="confidence", min_threshold=min_conf)

rules_fp = rules_fp.assign(
    A=rules_fp["antecedents"].apply(set),
    B=rules_fp["consequents"].apply(set),
)[['A', 'B', 'support', 'confidence', 'lift']].sort_values(
    by=["confidence", "lift", "support"],
    ascending=False,
).reset_index(drop=True)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", 0,
):
    display(rules_fp)

,A,B,support,confidence,lift
0,"{ROSES REGENCY TEACUP AND SAUCER, PINK REGENCY TEACUP AND SAUCER}",{GREEN REGENCY TEACUP AND SAUCER},0.022458,0.894137,20.679346
1,"{GREEN REGENCY TEACUP AND SAUCER, PINK REGENCY TEACUP AND SAUCER}",{ROSES REGENCY TEACUP AND SAUCER},0.022458,0.852484,18.606996
2,{PINK REGENCY TEACUP AND SAUCER},{GREEN REGENCY TEACUP AND SAUCER},0.026344,0.803995,18.594571
3,{PINK REGENCY TEACUP AND SAUCER},{ROSES REGENCY TEACUP AND SAUCER},0.025117,0.766542,16.731144
4,{GREEN REGENCY TEACUP AND SAUCER},{ROSES REGENCY TEACUP AND SAUCER},0.032071,0.741722,16.189404
5,{GARDENERS KNEELING PAD CUP OF TEA},{GARDENERS KNEELING PAD KEEP CALM},0.022458,0.717647,18.986580
6,"{ROSES REGENCY TEACUP AND SAUCER, GREEN REGENCY TEACUP AND SAUCER}",{PINK REGENCY TEACUP AND SAUCER},0.022458,0.700255,21.371331
7,{ROSES REGENCY TEACUP AND SAUCER},{GREEN REGENCY TEACUP AND SAUCER},0.032071,0.700000,16.189404
8,{CHARLOTTE BAG PINK POLKADOT},{RED RETROSPOT CHARLOTTE BAG},0.021517,0.692105,16.113529
9,{PINK REGENCY TEACUP AND SAUCER},"{ROSES REGENCY TEACUP AND SAUCER, GREEN REGENCY TEACUP AND SAUCER}",0.022458,0.685393,21.371331


In [7]:
basket = set(["ROSES REGENCY TEACUP AND SAUCER", "PINK REGENCY TEACUP AND SAUCER"])

rules_fp[rules_fp["A"] == basket]

,A,B,support,confidence,lift
0,"{ROSES REGENCY TEACUP AND SAUCER, PINK REGENCY...",{GREEN REGENCY TEACUP AND SAUCER},0.022458,0.894137,20.679346
